# OI Directional — Evaluation (notebooks_8/05)

Standalone model — no MFE filter.
Trades whenever OI directional model confidence exceeds threshold.

In [20]:
import pandas as pd
import numpy as np
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')
from pathlib import Path

FEAT6_DIR  = Path('../backend/data/features_6')
FEAT8_DIR  = Path('../backend/data/features_8')
PRICE_DIR  = Path('../backend/data/processed')
MODELS_DIR = Path('../backend/models_9/oi_directional')

MAJORS    = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'USDCAD', 'AUDUSD', 'NZDUSD']
TRAIN_END = '2024-06-30'

SPREADS = {
    'EURUSD': 0.2 * 0.0001,
    'GBPUSD': 0.4 * 0.0001,
    'USDJPY': 0.3 * 0.01 / 150,
    'USDCHF': 0.4 * 0.0001,
    'USDCAD': 0.4 * 0.0001,
    'AUDUSD': 0.3 * 0.0001,
    'NZDUSD': 0.5 * 0.0001,
}

print('Ready.')

Ready.


## 1. Load Model

In [21]:
bundle       = joblib.load(MODELS_DIR / 'model_oi_directional.joblib')
oi_feat_cols = bundle['feature_cols']
models       = bundle['models']

print(f'Features: {len(oi_feat_cols)}')
print(f'CV AUC:')
for k, v in bundle['cv_auc'].items():
    print(f'  {k}: {v:.4f}')


Features: 162
CV AUC:
  oi_4H: 0.7883
  oi_12H: 0.7896


## 2. Load Test Data & Predict

In [22]:
all_bars = []

for pair in MAJORS:
    print(f'  {pair}...', flush=True)

    df6 = pd.read_parquet(FEAT6_DIR / f'{pair}_features.parquet')
    df8 = pd.read_parquet(FEAT8_DIR / f'{pair}_geometric.parquet')

    df6.drop(columns=['pair'], errors='ignore', inplace=True)
    df8.drop(columns=['pair'], errors='ignore', inplace=True)

    idx = df6.index.intersection(df8.index)
    df  = pd.concat([df6.loc[idx], df8.loc[idx]], axis=1)
    df  = df.loc[:, ~df.columns.duplicated()]
    del df6, df8; gc.collect()

    df = df[df.index > TRAIN_END]

    price_df = pd.read_parquet(PRICE_DIR / f'{pair}_1H.parquet')
    close = price_df['close'].reindex(df.index)
    del price_df

    num_cols = [c for c in df.columns if df[c].dtype != object]
    df[num_cols] = df[num_cols].ffill().fillna(0).astype(np.float32)

    X = np.zeros((len(df), len(oi_feat_cols)), dtype=np.float32)
    for i, c in enumerate(oi_feat_cols):
        if c in df.columns:
            X[:, i] = df[c].values

    rows = pd.DataFrame(index=df.index)
    for name in ['oi_4H', 'oi_12H']:
        probs = models[name].predict_proba(X)
        rows[f'{name}_down'] = probs[:, 0]
        rows[f'{name}_flat'] = probs[:, 1]
        rows[f'{name}_up']   = probs[:, 2]
    del X; gc.collect()

    rows['pair']   = pair
    rows['close']  = close.values
    rows['spread'] = SPREADS[pair]
    all_bars.append(rows)
    del df, rows; gc.collect()

bars = pd.concat(all_bars).sort_index(); del all_bars; gc.collect()
nm = (bars.index.max() - bars.index.min()).days / 30
print(f'\nTest bars: {len(bars):,}  ({nm:.1f} months)')

  EURUSD...
  GBPUSD...
  USDJPY...
  USDCHF...
  USDCAD...
  AUDUSD...
  NZDUSD...

Test bars: 63,637  (18.3 months)


## 3. Threshold Sweep

In [23]:
def simulate(bars, horizon_col, thresh, nm):
    h = int(horizon_col.split('_')[1].replace('H',''))
    sim_rows = []
    for pair in sorted(bars['pair'].unique()):
        df_pair = bars[bars['pair'] == pair].sort_index()
        spread  = df_pair['spread'].iloc[0]
        in_trade_until = pd.Timestamp.min
        for k in range(len(df_pair)):
            ts  = df_pair.index[k]
            if ts < in_trade_until: continue
            row = df_pair.iloc[k]
            p_up   = row[f'{horizon_col}_up']
            p_down = row[f'{horizon_col}_down']
            if p_up > thresh and p_up > p_down:
                direction = 'long'
            elif p_down > thresh and p_down > p_up:
                direction = 'short'
            else:
                continue
            exit_ts    = ts + pd.Timedelta(hours=h)
            future     = df_pair[df_pair.index >= exit_ts]
            exit_close = future['close'].iloc[0] if len(future) > 0 else df_pair['close'].iloc[-1]
            log_ret    = np.log(exit_close / row['close'])
            pnl        = (log_ret if direction == 'long' else -log_ret) - spread
            sim_rows.append({'pnl': pnl, 'direction': direction, 'pair': pair, 'ts': ts})
            in_trade_until = exit_ts
    if not sim_rows: return None
    s = pd.DataFrame(sim_rows)
    pnl = s['pnl']
    ev = pnl.mean(); wr = (pnl > 0).mean()
    sh = (ev / pnl.std()) * np.sqrt(252*24) if pnl.std() > 0 else 0
    return {'n': len(s), 'wr': wr, 'ev': ev, 'sh': sh, 'pmo': pnl.sum()/nm, 'df': s}

print(f'{"Horizon":<10} {"Thresh":>7} {"N":>7} {"N/mo":>6} {"WR":>7} {"EV":>10} {"Sharpe":>8} {"PnL/mo":>10}')
print('=' * 78)
sweep_results = {}
for horizon_col in ['oi_4H', 'oi_12H']:
    for thresh in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        r = simulate(bars, horizon_col, thresh, nm)
        if r is None: continue
        sweep_results[(horizon_col, thresh)] = r
        flag = ' <<<' if r['ev'] > 0 else ''
        print(f'{horizon_col:<10} {thresh:>7.2f} {r["n"]:>7,} {r["n"]/nm:>6.1f} {r["wr"]:>7.1%} {r["ev"]:>+10.6f} {r["sh"]:>+8.2f} {r["pmo"]:>+10.4f}{flag}')
    print()


Horizon     Thresh       N   N/mo      WR         EV   Sharpe     PnL/mo
oi_4H         0.35  13,013  712.4   49.7%  +0.000024    +0.56    +0.0174 <<<
oi_4H         0.40  12,050  659.7   50.3%  +0.000053    +1.16    +0.0349 <<<
oi_4H         0.45  10,987  601.5   50.8%  +0.000085    +1.64    +0.0512 <<<
oi_4H         0.50   9,778  535.3   50.5%  +0.000100    +1.86    +0.0535 <<<
oi_4H         0.55   8,625  472.2   50.4%  +0.000087    +1.56    +0.0412 <<<
oi_4H         0.60   7,430  406.8   49.8%  +0.000086    +1.47    +0.0350 <<<

oi_12H        0.35   5,120  280.3   50.3%  -0.000024    -0.51    -0.0068
oi_12H        0.40   4,962  271.6   50.6%  -0.000021    -0.40    -0.0056
oi_12H        0.45   4,780  261.7   49.7%  -0.000043    -0.83    -0.0112
oi_12H        0.50   4,563  249.8   50.1%  -0.000047    -0.86    -0.0117
oi_12H        0.55   4,319  236.4   50.3%  +0.000030    +0.58    +0.0070 <<<
oi_12H        0.60   3,968  217.2   49.9%  +0.000035    +0.58    +0.0077 <<<



## 4. Best Config Deep Dive

In [24]:
# Pick best by Sharpe among positive EV configs
best_key = max(
    [(k, v) for k, v in sweep_results.items() if v['ev'] > 0],
    key=lambda x: x[1]['sh']
)[0]
BEST_HORIZON, BEST_THRESH = best_key
best = sweep_results[best_key]
print(f'Best config: {BEST_HORIZON}  thresh={BEST_THRESH}')
print(f'  N={best["n"]:,}  WR={best["wr"]:.1%}  EV={best["ev"]:+.6f}  Sharpe={best["sh"]:+.2f}  PnL/mo={best["pmo"]:+.4f}')

sub = best['df'].set_index('ts')
pnl = sub['pnl']

print(f'\nPER-PAIR:')
print(f'  {"Pair":<10} {"N":>6} {"/mo":>5} {"WR":>7} {"EV":>10} {"PnL/mo":>10}')
print('  ' + '-'*52)
for pair, g in sub.groupby('pair'):
    wr  = (g['pnl']>0).mean(); ev = g['pnl'].mean()
    pmo = g['pnl'].sum()/nm;   npm = len(g)/nm
    flag = ' <<<' if ev > 0 else ''
    print(f'  {pair:<10} {len(g):>6,} {npm:>5.1f} {wr:>7.1%} {ev:>+10.6f} {pmo:>+10.4f}{flag}')

print(f'\nBY DIRECTION:')
for d, g in sub.groupby('direction'):
    wr = (g['pnl']>0).mean(); ev = g['pnl'].mean()
    print(f'  {d:<6} N={len(g):,}  WR={wr:.1%}  EV={ev:+.6f}  PnL/mo={g["pnl"].sum()/nm:+.4f}')

print(f'\nMONTHLY:')
print(f'  {"Month":<10} {"N":>5} {"WR":>7} {"EV":>10} {"CumPnL":>10}')
print('  ' + '-'*48)
cum = 0
for (yr, mo), g in sub.groupby([sub.index.year, sub.index.month]):
    ev = g['pnl'].mean(); cum += g['pnl'].sum()
    wr = (g['pnl']>0).mean()
    flag = ' <--' if ev < 0 else ''
    print(f'  {yr}-{mo:02d}    {len(g):>5} {wr:>7.1%} {ev:>+10.6f} {cum:>+10.4f}{flag}')


Best config: oi_4H  thresh=0.5
  N=9,778  WR=50.5%  EV=+0.000100  Sharpe=+1.86  PnL/mo=+0.0535

PER-PAIR:
  Pair            N   /mo      WR         EV     PnL/mo
  ----------------------------------------------------
  AUDUSD      1,418  77.6   50.4%  -0.000052    -0.0041
  EURUSD      1,394  76.3   50.2%  +0.000023    +0.0018 <<<
  GBPUSD      1,371  75.1   51.4%  +0.000040    +0.0030 <<<
  NZDUSD      1,378  75.4   50.9%  +0.000423    +0.0319 <<<
  USDCAD      1,388  76.0   51.7%  +0.000348    +0.0264 <<<
  USDCHF      1,447  79.2   49.1%  -0.000038    -0.0030
  USDJPY      1,382  75.7   50.0%  -0.000034    -0.0026

BY DIRECTION:
  long   N=4,209  WR=51.3%  EV=+0.000207  PnL/mo=+0.0477
  short  N=5,569  WR=49.9%  EV=+0.000019  PnL/mo=+0.0058

MONTHLY:
  Month          N      WR         EV     CumPnL
  ------------------------------------------------
  2024-07      590   52.4%  -0.000039    -0.0232 <--
  2024-08      550   49.1%  +0.000002    -0.0221
  2024-09      510   50.8%  +0.000

In [25]:
# Verify OI prediction accuracy on test set
import pandas as pd
import numpy as np
from pathlib import Path
import gc

FEAT6_DIR     = Path('../backend/data/features_6')
OI_STD_WINDOW = 24

results = []

for pair in MAJORS:
    df6 = pd.read_parquet(FEAT6_DIR / f'{pair}_features.parquet')
    df6.drop(columns=['pair'], errors='ignore', inplace=True)
    df6 = df6[df6.index > TRAIN_END]

    oi     = df6['order_imbalance']
    oi_std = oi.rolling(OI_STD_WINDOW).std()

    n = len(df6)
    for i in range(n - 4):
        if pd.isna(oi.iloc[i]) or pd.isna(oi_std.iloc[i]) or oi_std.iloc[i] < 1e-10:
            continue
        std  = oi_std.iloc[i]
        oi0  = oi.iloc[i]
        delta = oi.iloc[i + 4] - oi0
        true_cls = 2 if delta > std else (0 if delta < -std else 1)
        results.append({'pair': pair, 'ts': df6.index[i], 'true_4H': true_cls})

    del df6; gc.collect()

df_true = pd.DataFrame(results).set_index('ts')

# merge with model predictions
df_check = bars[['pair', 'oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].copy()
df_check['pred_cls'] = np.argmax(df_check[['oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].values, axis=1)
df_check['pred_conf'] = df_check[['oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].max(axis=1)
df_check = df_check.join(df_true['true_4H'], how='inner')

print(f'OI prediction accuracy (oi_4H) on test set:')
print(f'  Overall accuracy: {(df_check["pred_cls"] == df_check["true_4H"]).mean():.1%}  (n={len(df_check):,})')
print()

print(f'  By confidence threshold:')
print(f'  {"Conf>":<6} {"N":>7} {"Acc":>7} {"Acc(0)":>8} {"Acc(2)":>8}')
print('  ' + '-'*40)
for thresh in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    mask = df_check['pred_conf'] > thresh
    sub  = df_check[mask]
    if len(sub) < 10: continue
    acc    = (sub['pred_cls'] == sub['true_4H']).mean()
    acc_dn = (sub[sub['pred_cls']==0]['true_4H'] == 0).mean() if (sub['pred_cls']==0).sum() > 0 else float('nan')
    acc_up = (sub[sub['pred_cls']==2]['true_4H'] == 2).mean() if (sub['pred_cls']==2).sum() > 0 else float('nan')
    print(f'  {thresh:<6.2f} {len(sub):>7,} {acc:>7.1%} {acc_dn:>8.1%} {acc_up:>8.1%}')

print()
print(f'  Base rates — down:{(df_check["true_4H"]==0).mean():.1%}  flat:{(df_check["true_4H"]==1).mean():.1%}  up:{(df_check["true_4H"]==2).mean():.1%}')


OI prediction accuracy (oi_4H) on test set:
  Overall accuracy: 47.3%  (n=443,359)

  By confidence threshold:
  Conf>        N     Acc   Acc(0)   Acc(2)
  ----------------------------------------
  0.35   443,359   47.3%    32.2%    32.3%
  0.40   443,331   47.3%    32.2%    32.3%
  0.45   442,710   47.3%    32.2%    32.3%
  0.50   411,998   47.8%    32.7%    32.6%
  0.55   335,619   48.7%    33.9%    33.0%
  0.60   244,650   48.9%    35.0%    33.7%

  Base rates — down:24.0%  flat:51.7%  up:24.2%


In [26]:
print(f'bars rows: {len(df_check)}')
print(f'df_true rows: {len(df_true)}')
print(f'joined rows: {len(df_check.dropna(subset=["true_4H"]))}')
print(f'Sample pred vs true:')
print(df_check[['pair', 'pred_cls', 'pred_conf', 'true_4H']].head(20))


bars rows: 443359
df_true rows: 64762
joined rows: 443359
Sample pred vs true:
                       pair  pred_cls  pred_conf  true_4H
datetime                                                 
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCHF         1   0.612501        1
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCHF         1   0.612501        2
2024-07-01 20:00:00  USDCAD         1   0.772266        2
2024-07-01 20:00:00  USDCAD         1   0.772266        1
2024-07-01 20:00:00  USDCAD         1   0.772266        2
2024-07-01 20:00:00  USDCAD         1   0.772266        2
2024-07-01 20:00:00  USDCAD         1   0.772266        2
2024-07-01 20:00:00  USDCAD         1   0.772266        2
2024-07-01 20:00:00  USDCAD         1   0.772266   

In [28]:
df_check = bars[['pair', 'oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].copy()
df_check['pred_cls']  = np.argmax(df_check[['oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].values, axis=1)
df_check['pred_conf'] = df_check[['oi_4H_down', 'oi_4H_flat', 'oi_4H_up']].max(axis=1)
df_check.index.name = 'datetime'

df_true.index.name = 'datetime'

df_check = df_check.reset_index().merge(
    df_true.reset_index(),
    on=['datetime', 'pair'],
    how='inner'
).set_index('datetime')

print(f'Matched rows: {len(df_check):,}')
print(f'\nOI prediction accuracy (oi_4H) on test set:')
print(f'  Overall accuracy: {(df_check["pred_cls"] == df_check["true_4H"]).mean():.1%}')
print()
print(f'  {"Conf>":<6} {"N":>7} {"Acc":>7} {"Acc(0)":>8} {"Acc(2)":>8}')
print('  ' + '-'*40)
for thresh in [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    mask = df_check['pred_conf'] > thresh
    sub  = df_check[mask]
    if len(sub) < 10: continue
    acc    = (sub['pred_cls'] == sub['true_4H']).mean()
    acc_dn = (sub[sub['pred_cls']==0]['true_4H'] == 0).mean() if (sub['pred_cls']==0).sum() > 0 else float('nan')
    acc_up = (sub[sub['pred_cls']==2]['true_4H'] == 2).mean() if (sub['pred_cls']==2).sum() > 0 else float('nan')
    print(f'  {thresh:<6.2f} {len(sub):>7,} {acc:>7.1%} {acc_dn:>8.1%} {acc_up:>8.1%}')

print()
print(f'  Base rates — down:{(df_check["true_4H"]==0).mean():.1%}  flat:{(df_check["true_4H"]==1).mean():.1%}  up:{(df_check["true_4H"]==2).mean():.1%}')


Matched rows: 63,469

OI prediction accuracy (oi_4H) on test set:
  Overall accuracy: 63.1%

  Conf>        N     Acc   Acc(0)   Acc(2)
  ----------------------------------------
  0.35    63,469   63.1%    64.3%    70.8%
  0.40    63,465   63.1%    64.3%    70.8%
  0.45    63,376   63.1%    64.3%    70.8%
  0.50    58,981   64.1%    66.2%    72.3%
  0.55    48,049   66.8%    70.4%    75.5%
  0.60    35,036   69.7%    73.8%    78.6%

  Base rates — down:24.0%  flat:51.7%  up:24.2%


In [30]:
# Does OI direction actually predict price direction?
price_results = []

for pair in MAJORS:
    df6 = pd.read_parquet(FEAT6_DIR / f'{pair}_features.parquet')
    df6.drop(columns=['pair'], errors='ignore', inplace=True)

    price_df = pd.read_parquet(PRICE_DIR / f'{pair}_1H.parquet')
    close = price_df['close'].reindex(df6.index)

    oi     = df6['order_imbalance']
    oi_std = oi.rolling(OI_STD_WINDOW).std()

    oi    = oi[oi.index > TRAIN_END]
    oi_std = oi_std[oi_std.index > TRAIN_END]
    close  = close[close.index > TRAIN_END]

    for i in range(len(oi) - 4):
        if pd.isna(oi.iloc[i]) or pd.isna(oi_std.iloc[i]) or oi_std.iloc[i] < 1e-10:
            continue
        std = oi_std.iloc[i]; oi0 = oi.iloc[i]
        delta_oi = oi.iloc[i + 4] - oi0
        true_oi_cls = 2 if delta_oi > std else (0 if delta_oi < -std else 1)
        if true_oi_cls == 1: continue

        p0 = close.iloc[i]; p4 = close.iloc[i + 4]
        if pd.isna(p0) or pd.isna(p4): continue
        price_ret = np.log(p4 / p0)
        price_dir = 1 if price_ret > 0 else 0
        price_results.append({'oi_cls': true_oi_cls, 'price_ret': price_ret, 'price_dir': price_dir})

    del df6, price_df; gc.collect()

df_pr = pd.DataFrame(price_results)
up  = df_pr[df_pr['oi_cls'] == 2]
dn  = df_pr[df_pr['oi_cls'] == 0]
print(f'When OI truly goes UP   (n={len(up):,}) — price up%: {up["price_dir"].mean():.1%}  avg ret: {up["price_ret"].mean():+.6f}')
print(f'When OI truly goes DOWN (n={len(dn):,}) — price up%: {dn["price_dir"].mean():.1%}  avg ret: {dn["price_ret"].mean():+.6f}')


When OI truly goes UP   (n=14,994) — price up%: 63.6%  avg ret: +0.000684
When OI truly goes DOWN (n=15,104) — price up%: 37.2%  avg ret: -0.000685


In [33]:
# At conf>0.60, what does price actually do on predicted bars
df_hc2 = df_check.reset_index().copy()
df_hc2 = df_hc2.merge(df_pr2, on=['datetime', 'pair'], how='inner')

sub = df_hc2[df_hc2['pred_conf'] > 0.60]
up  = sub[sub['pred_cls'] == 2]
dn  = sub[sub['pred_cls'] == 0]

print(f'Conf > 0.60  (n={len(sub):,})')
print(f'  Predicted UP   (n={len(up):,}): price up% = {up["price_dir"].mean():.1%}  avg ret = {up["price_ret"].mean():+.6f}')
print(f'  Predicted DOWN (n={len(dn):,}): price up% = {dn["price_dir"].mean():.1%}  (price dn% = {1-dn["price_dir"].mean():.1%})  avg ret = {(-dn["price_ret"]).mean():+.6f}')


Conf > 0.60  (n=16,371)
  Predicted UP   (n=3,689): price up% = 56.0%  avg ret = +0.000407
  Predicted DOWN (n=5,022): price up% = 44.5%  (price dn% = 55.5%)  avg ret = +0.000302
